In [39]:
import warnings
warnings.filterwarnings("ignore")
 
# ── Standard library ────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy.stats import spearmanr
 
# ── Data ────────────────────────────────────────────────────────────────────
import yfinance as yf
 
# ── Sklearn ─────────────────────────────────────────────────────────────────
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
 
# ── TensorFlow / Keras  (FIX 2: all layers imported explicitly) ─────────────
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    LSTM, Dense, Dropout, BatchNormalization, Input
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

In [28]:
# ─────────────────────────────────────────────
# 0. Reproducibility
# ─────────────────────────────────────────────
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [29]:
# ─────────────────────────────────────────────
# 1. Configuration
# ─────────────────────────────────────────────
TICKER      = "^NSEI"          # Nifty 50 on Yahoo Finance
START_DATE  = "2010-01-01"
END_DATE    = "2024-12-31"
LOOKBACK    = 60               # sequence length (trading days)
FORECAST_H  = 1                # 1-day-ahead forecast
TEST_RATIO  = 0.15
VAL_RATIO   = 0.10
EPOCHS      = 100
BATCH_SIZE  = 64
LSTM_UNITS  = [128, 64]        # stacked LSTM layer sizes
DROPOUT     = 0.25
LR          = 1e-3

In [40]:
# ─────────────────────────────────────────────
# 2. Data Download
# ─────────────────────────────────────────────
print("▶ Downloading Nifty 50 data …")
raw = yf.download(TICKER, start=START_DATE, end=END_DATE, auto_adjust=True)
 
if raw.empty:
    raise ValueError("No data returned. Check ticker / internet connection.")
 
df = raw[["Open", "High", "Low", "Close", "Volume"]].copy()
 
# flatten MultiIndex columns that yfinance sometimes returns
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)
 
df.dropna(inplace=True)

▶ Downloading Nifty 50 data …


[*********************100%***********************]  1 of 1 completed


In [41]:
# ─────────────────────────────────────────────
# 3. Feature Engineering
# ─────────────────────────────────────────────
df["log_ret"]  = np.log(df["Close"] / df["Close"].shift(1))
df["range"]    = np.log(df["High"]  / df["Low"])
df["gap"]      = np.log(df["Open"]  / df["Close"].shift(1))
df["vol_5d"]   = df["log_ret"].rolling(5).std()
df["vol_21d"]  = df["log_ret"].rolling(21).std()
df["mom_5d"]   = df["log_ret"].rolling(5).sum()
df["mom_21d"]  = df["log_ret"].rolling(21).sum()
 
def compute_rsi(series, period=14):
    delta = series.diff()
    gain  = delta.clip(lower=0).rolling(period).mean()
    loss  = (-delta.clip(upper=0)).rolling(period).mean()
    rs    = gain / (loss + 1e-9)
    return 100 - 100 / (1 + rs)
 
df["rsi_14"]   = compute_rsi(df["Close"])
df["vol_chg"]  = np.log(df["Volume"] / df["Volume"].shift(1))
 
# ── FIX 1: scrub inf/nan right after feature engineering ────────────────────
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.ffill(inplace=True)
df.dropna(inplace=True)
 
FEATURE_COLS = [
    "log_ret", "range", "gap",
    "vol_5d",  "vol_21d",
    "mom_5d",  "mom_21d",
    "rsi_14",  "vol_chg",
]
TARGET_COL = "log_ret"
 
print(f"   Dataset: {len(df)} rows | {len(FEATURE_COLS)} features")

   Dataset: 2931 rows | 9 features


In [42]:

# ─────────────────────────────────────────────
# 4. Chronological Train / Val / Test Split
# ─────────────────────────────────────────────
n       = len(df)
n_test  = int(n * TEST_RATIO)
n_val   = int(n * VAL_RATIO)
n_train = n - n_val - n_test
 
df_train = df.iloc[:n_train].copy()
df_val   = df.iloc[n_train : n_train + n_val].copy()
df_test  = df.iloc[n_train + n_val:].copy()
 
print(f"   Train: {len(df_train)} | Val: {len(df_val)} | Test: {len(df_test)}")

   Train: 2199 | Val: 293 | Test: 439


In [ ]:
# ─────────────────────────────────────────────
# 5. Scaling  (fit ONLY on train — no leakage)
# ─────────────────────────────────────────────
scaler = MinMaxScaler(feature_range=(-1, 1))
scaler.fit(df_train[FEATURE_COLS])
 
train_scaled = scaler.transform(df_train[FEATURE_COLS])
val_scaled   = scaler.transform(df_val[FEATURE_COLS])
test_scaled  = scaler.transform(df_test[FEATURE_COLS])
 
target_idx = FEATURE_COLS.index(TARGET_COL)
 
# Sanity check — will raise immediately if any inf/nan slipped through
for name, arr in [("train", train_scaled), ("val", val_scaled), ("test", test_scaled)]:
    assert not np.any(np.isinf(arr)), f"inf present in {name} after scaling"
    assert not np.any(np.isnan(arr)), f"nan present in {name} after scaling"

In [44]:
# ─────────────────────────────────────────────
# 6. Sequence Builder
# ─────────────────────────────────────────────
def build_sequences(data_scaled, lookback, target_idx, forecast_h=1):
    """
    Returns
        X : (samples, lookback, n_features)
        y : (samples,)   scaled target value at t+forecast_h
    """
    X, y = [], []
    for i in range(lookback, len(data_scaled) - forecast_h + 1):
        X.append(data_scaled[i - lookback : i])
        y.append(data_scaled[i + forecast_h - 1, target_idx])
    return np.array(X), np.array(y)
 
X_train, y_train = build_sequences(train_scaled, LOOKBACK, target_idx, FORECAST_H)
 
# Prepend the last LOOKBACK rows of the preceding split for continuity
X_val, y_val = build_sequences(
    np.vstack([train_scaled[-LOOKBACK:], val_scaled]),
    LOOKBACK, target_idx, FORECAST_H,
)
X_test, y_test = build_sequences(
    np.vstack([val_scaled[-LOOKBACK:], test_scaled]),
    LOOKBACK, target_idx, FORECAST_H,
)
 
print(f"   Sequences — Train {X_train.shape} | Val {X_val.shape} | Test {X_test.shape}")

   Sequences — Train (2139, 60, 9) | Val (293, 60, 9) | Test (439, 60, 9)


In [45]:
# ─────────────────────────────────────────────
# 7. Model
# ─────────────────────────────────────────────
def build_model(lookback, n_features, lstm_units, dropout, lr):
    model = Sequential([
        Input(shape=(lookback, n_features)),
        LSTM(lstm_units[0], return_sequences=True),
        BatchNormalization(),
        Dropout(dropout),
        LSTM(lstm_units[1], return_sequences=False),
        BatchNormalization(),
        Dropout(dropout),
        Dense(32, activation="relu"),
        Dense(1),
    ])
    model.compile(optimizer=Adam(lr), loss="huber", metrics=["mae"])
    return model
 
model = build_model(LOOKBACK, len(FEATURE_COLS), LSTM_UNITS, DROPOUT, LR)
model.summary()
 
# ─────────────────────────────────────────────
# 8. Training
# ─────────────────────────────────────────────
callbacks = [
    EarlyStopping(patience=12, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(factor=0.5, patience=6, min_lr=1e-6, verbose=1),
]
 
print("\n▶ Training …")
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=1,
)

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_9 (LSTM)                   │ (None, 60, 128)        │        70,656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_8           │ (None, 60, 128)        │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_8 (Dropout)             │ (None, 60, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_10 (LSTM)                  │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_9           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_9 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 122,945 (480.25 KB)

 Trainable params: 122,561 (478.75 KB)

 Non-trainable params: 384 (1.50 KB)


▶ Training …
Epoch 1/100
34/34 ━━━━━━━━━━━━━━━━━━━━ 14s 189ms/step - loss: 0.1972 - mae: 0.4931 - val_loss: 0.0100 - val_mae: 0.1152 - learning_rate: 0.0010
Epoch 2/100
34/34 ━━━━━━━━━━━━━━━━━━━━ 8s 130ms/step - loss: 0.0952 - mae: 0.3352 - val_loss: 0.0107 - val_mae: 0.1224 - learning_rate: 0.0010
Epoch 3/100
34/34 ━━━━━━━━━━━━━━━━━━━━ 5s 133ms/step - loss: 0.0652 - mae: 0.2758 - val_loss: 0.0088 - val_mae: 0.1091 - learning_rate: 0.0010
Epoch 4/100
34/34 ━━━━━━━━━━━━━━━━━━━━ 4s 125ms/step - loss: 0.0457 - mae: 0.2319 - val_loss: 0.0076 - val_mae: 0.0998 - learning_rate: 0.0010
Epoch 5/100
34/34 ━━━━━━━━━━━━━━━━━━━━ 4s 121ms/step - loss: 0.0350 - mae: 0.2016 - val_loss: 0.0071 - val_mae: 0.0950 - learning_rate: 0.0010
Epoch 6/100
34/34 ━━━━━━━━━━━━━━━━━━━━ 4s 130ms/step - loss: 0.0280 - mae: 0.1808 - val_loss: 0.0056 - val_mae: 0.0821 - learning_rate: 0.0010
Epoch 7/100
34/34 ━━━━━━━━━━━━━━━━━━━━ 4s 120ms/step - loss: 0.0245 - mae: 0.1685 - val_loss: 0.0051 - val_mae: 0.0780 - learni

In [46]:
# ─────────────────────────────────────────────
# 9. Predictions & Inverse Scaling
# ─────────────────────────────────────────────
def inverse_target(scaled_vals, scaler, n_features, target_idx):
    dummy = np.zeros((len(scaled_vals), n_features))
    dummy[:, target_idx] = np.array(scaled_vals).ravel()
    return scaler.inverse_transform(dummy)[:, target_idx]
 
n_feat = len(FEATURE_COLS)
 
# FIX 4: ravel() immediately → guaranteed 1-D arrays everywhere
y_pred = inverse_target(
    model.predict(X_test, verbose=0), scaler, n_feat, target_idx
).ravel()
y_true = inverse_target(y_test, scaler, n_feat, target_idx).ravel()
 

In [47]:
# ─────────────────────────────────────────────
# 10. Metrics
# ─────────────────────────────────────────────
rmse    = np.sqrt(mean_squared_error(y_true, y_pred))
mae     = mean_absolute_error(y_true, y_pred)
corr    = np.corrcoef(y_true, y_pred)[0, 1]
dir_acc = np.mean(np.sign(y_true) == np.sign(y_pred))
ic, _   = spearmanr(y_true, y_pred)
 
print("\n══════════════════════════════════════════")
print("  Test-Set Performance")
print("══════════════════════════════════════════")
print(f"  RMSE               : {rmse:.6f}")
print(f"  MAE                : {mae:.6f}")
print(f"  Pearson Corr       : {corr:.4f}")
print(f"  Directional Acc    : {dir_acc:.2%}")
print(f"  Spearman IC        : {ic:.4f}")
print("══════════════════════════════════════════\n")
 


══════════════════════════════════════════
  Test-Set Performance
══════════════════════════════════════════
  RMSE               : 0.008524
  MAE                : 0.006416
  Pearson Corr       : -0.0266
  Directional Acc    : 44.65%
  Spearman IC        : -0.0517
══════════════════════════════════════════



In [ ]:
combined_index = pd.DatetimeIndex(
    list(df_val.index[-LOOKBACK:]) + list(df_test.index)
)
test_dates = combined_index[LOOKBACK : LOOKBACK + len(y_true)]
assert len(test_dates) == len(y_true), (
    f"Date/array mismatch: {len(test_dates)} vs {len(y_true)}"
)
 
# ── Theme ────────────────────────────────────────────────────────────────────
BG_DARK  = "#0d1117"
BG_PANEL = "#161b22"
TICK_CLR = "#8b949e"
SPINE    = "#30363d"
TEXT     = "#e6edf3"
GOLD     = "#f0b429"
CYAN     = "#39d0d8"
MAGENTA  = "#e040fb"
GREEN    = "#3fb950"
RED      = "#f85149"
 
fig, axes = plt.subplots(3, 2, figsize=(16, 14))
fig.patch.set_facecolor(BG_DARK)
 
for ax in axes.ravel():
    ax.set_facecolor(BG_PANEL)
    ax.tick_params(colors=TICK_CLR)
    ax.yaxis.label.set_color(TICK_CLR)
    ax.xaxis.label.set_color(TICK_CLR)
    for spine in ax.spines.values():
        spine.set_edgecolor(SPINE)
 
def style_legend(ax):
    ax.legend(facecolor=BG_PANEL, labelcolor=TEXT,
              framealpha=0.9, edgecolor=SPINE)
    

    ax = axes[0, 0]
ax.plot(history.history["loss"],     color=CYAN,    lw=1.5, label="Train Loss")
ax.plot(history.history["val_loss"], color=MAGENTA, lw=1.5, ls="--", label="Val Loss")
ax.set_title("Training vs Validation Loss", color=TEXT, fontsize=11, pad=8)
ax.set_xlabel("Epoch")
style_legend(ax)


ax = axes[0, 1]
ax.plot(test_dates, y_true, color=GREEN, lw=0.9, alpha=0.85, label="Actual")
ax.plot(test_dates, y_pred, color=GOLD,  lw=0.9, alpha=0.85, ls="--", label="Predicted")
ax.axhline(0, color=SPINE, lw=0.8)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.set_title("Log Return: Actual vs Predicted", color=TEXT, fontsize=11, pad=8)
ax.set_xlabel("Date")
style_legend(ax)
 
# ── 11c. Scatter ─────────────────────────────────────────────────────────────
ax = axes[1, 0]
ax.scatter(y_true, y_pred, alpha=0.35, s=8, color=CYAN)
lim = max(np.abs(y_true).max(), np.abs(y_pred).max()) * 1.1
ax.plot([-lim, lim], [-lim, lim], color=RED, lw=1, ls="--")
ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
ax.set_xlabel("Actual"); ax.set_ylabel("Predicted")
ax.set_title(f"Scatter  (Pearson r = {corr:.3f})", color=TEXT, fontsize=11, pad=8)
 
# ── 11d. Residuals over time ──────────────────────────────────────────────────
ax = axes[1, 1]
residuals  = y_true - y_pred
bar_colors = [GREEN if r >= 0 else RED for r in residuals]
n_pts      = len(residuals)
# Use integer x-axis to avoid DatetimeIndex bar-width issues
ax.bar(range(n_pts), residuals, color=bar_colors, width=1.0, alpha=0.75)
ax.axhline(0, color=TICK_CLR, lw=0.8)
tick_pos    = np.linspace(0, n_pts - 1, 5, dtype=int)
tick_labels = [test_dates[i].strftime("%Y-%m") for i in tick_pos]
ax.set_xticks(tick_pos)
ax.set_xticklabels(tick_labels, color=TICK_CLR, fontsize=8)
ax.set_title("Prediction Residuals", color=TEXT, fontsize=11, pad=8)
ax.set_xlabel("Date")
 
# ── 11e. Residual distribution ────────────────────────────────────────────────
ax = axes[2, 0]
ax.hist(residuals, bins=50, color=MAGENTA, alpha=0.75, edgecolor="none")
ax.axvline(0, color=GOLD, lw=1.5, ls="--")
ax.set_title("Residual Distribution", color=TEXT, fontsize=11, pad=8)
ax.set_xlabel("Residual")
 
# ── 11f. Cumulative P&L vs buy-and-hold ──────────────────────────────────────
ax = axes[2, 1]
cum_bh    = np.cumsum(y_true)
cum_strat = np.cumsum(y_true * np.sign(y_pred))
ax.plot(test_dates, cum_bh,    color=CYAN, lw=1.5, label="Buy & Hold")
ax.plot(test_dates, cum_strat, color=GOLD, lw=1.5, label="LSTM Strategy")
ax.axhline(0, color=SPINE, lw=0.8)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.set_title(f"Cumulative Log Return  (Dir Acc {dir_acc:.1%})", color=TEXT, fontsize=11, pad=8)
ax.set_xlabel("Date")
style_legend(ax)
 
fig.suptitle("Nifty 50 — LSTM Return Forecast", color=TEXT,
             fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("nifty50_lstm_results.png", dpi=150, bbox_inches="tight",
            facecolor=fig.get_facecolor())
print("▶ Plots saved → nifty50_lstm_results.png")
plt.show()
 

In [52]:
# ─────────────────────────────────────────────
# 12. Save model
# ─────────────────────────────────────────────
model.save("nifty50_lstm_model.keras")
print("▶ Model saved → nifty50_lstm_model.keras")

▶ Model saved → nifty50_lstm_model.keras


In [53]:
last_window     = test_scaled[-LOOKBACK:].reshape(1, LOOKBACK, len(FEATURE_COLS))
next_ret_scaled = model.predict(last_window, verbose=0).ravel()
next_ret        = inverse_target(next_ret_scaled, scaler, n_feat, target_idx)[0]
 
print(f"\n▶ Next-day log return forecast : {next_ret:+.5f}")
print(f"   Implied price direction      : {'↑ UP' if next_ret > 0 else '↓ DOWN'}")
print(f"   Last close (Nifty 50)        : {df['Close'].iloc[-1]:.2f}")
 


▶ Next-day log return forecast : -0.00409
   Implied price direction      : ↓ DOWN
   Last close (Nifty 50)        : 23644.90
